## UAV Intrusion Detection Stacked Ensemble
This notebook uses One-Class SVM, Autoencoder, and Local Outlier Factor as base models, and combines their outputs with a meta-classifier ensemble.

In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import evaluate, summary, time_inference, time_inference_torch
from gps_ids import protocol
from gps_ids.results import append_validated

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import classification_report, accuracy_score, f1_score
import os
import gc
import time

In [ ]:
# Constants
RANDOM_STATE = 1082
MODEL_NAME = "StackedEnsemble_RF"

SYNTH_GENERATORS = ["ctgan", "gc", "tvae"]
SAMPLE_SIZES = [10_000, 100_000, 1_000_000]
SYNTH_FILES = [f"spoofing_{g}_{n}.csv" for g in SYNTH_GENERATORS for n in SAMPLE_SIZES]


In [4]:
results = []

In [ ]:
import gc

results = []
for synth_name in SYNTH_FILES:
    try:
        EXP = protocol.synthetic_ensemble_vs_full_real("2d", synth_name)
    except FileNotFoundError:
        print(f"Skipping {synth_name}: file not found")
        continue
    if int((EXP.y_train == 0).sum()) < 10:
        print(f"Too few benign samples in {synth_name}, skipping.")
        continue

    scaler = EXP.scaler
    X_train_scaled = EXP.X_fit.values
    X_synth_full_scaled = EXP.X_train.values
    X_real_scaled = EXP.X_test.values
    y_synth_full = EXP.y_train
    y_real = EXP.y_test

    # PCA for OC-SVM and LOF (85% variance)
    pca_svm_lof = PCA(n_components=0.85, random_state=RANDOM_STATE)
    X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
    X_synth_full_pca_svm_lof = pca_svm_lof.transform(X_synth_full_scaled)
    X_real_pca_svm_lof = pca_svm_lof.transform(X_real_scaled)

    # PCA for Autoencoder (95% variance)
    pca_ae = PCA(n_components=0.95, random_state=RANDOM_STATE)
    X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
    X_synth_full_pca_ae = pca_ae.transform(X_synth_full_scaled)
    X_real_pca_ae = pca_ae.transform(X_real_scaled)

    # Free up scaled arrays
    del X_train_scaled, X_synth_full_scaled, X_real_scaled
    gc.collect()

    # One-Class SVM
    start_ocsvm = time.time()
    ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
    ocsvm.fit(X_train_pca_svm_lof)
    ocsvm_train_time = time.time() - start_ocsvm
    ocsvm_score_synth = ocsvm.decision_function(X_synth_full_pca_svm_lof)
    ocsvm_score_real = ocsvm.decision_function(X_real_pca_svm_lof)

    # Local Outlier Factor
    start_lof = time.time()
    lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
    lof.fit(X_train_pca_svm_lof)
    lof_train_time = time.time() - start_lof
    lof_score_synth = lof.decision_function(X_synth_full_pca_svm_lof)
    lof_score_real = lof.decision_function(X_real_pca_svm_lof)

    # Free up pca arrays for SVM/LOF
    del X_train_pca_svm_lof, X_synth_full_pca_svm_lof, X_real_pca_svm_lof
    gc.collect()

    # Autoencoder using MLP
    start_ae = time.time()
    ae = MLPRegressor(hidden_layer_sizes=(100, 50), activation='tanh', max_iter=1000, random_state=RANDOM_STATE, learning_rate='adaptive',early_stopping=True,)
    ae.fit(X_train_pca_ae, X_train_pca_ae)
    ae_train_time = time.time() - start_ae
    recon_synth = ae.predict(X_synth_full_pca_ae)
    recon_real = ae.predict(X_real_pca_ae)
    ae_score_synth = np.mean((X_synth_full_pca_ae - recon_synth) ** 2, axis=1)
    ae_score_real = np.mean((X_real_pca_ae - recon_real) ** 2, axis=1)

    # Free up pca arrays for AE and reconstructions
    del X_train_pca_ae, X_synth_full_pca_ae, X_real_pca_ae, recon_synth, recon_real
    gc.collect()

    # Stack anomaly scores as features
    stacked_features_synth = np.vstack([ocsvm_score_synth, lof_score_synth, ae_score_synth]).T
    stacked_features_real = np.vstack([ocsvm_score_real, lof_score_real, ae_score_real]).T

    # Meta-classifier (Random Forest) trained on synthetic, evaluated on real
    start_rf = time.time()
    meta_clf = RandomForestClassifier(random_state=RANDOM_STATE)
    meta_clf.fit(stacked_features_synth, y_synth_full)
    rf_train_time = time.time() - start_rf
    stacked_pred = meta_clf.predict(stacked_features_real)

    # Clean up stacked features and y

    report = classification_report(
        y_real,
        stacked_pred,
        labels=[0, 1, 2, 3],
        target_names=["benign", "attack1", "attack2", "attack3"],
        zero_division=0,
        output_dict=True
    )
    f1 = report["weighted avg"]["f1-score"]
    accuracy = accuracy_score(y_real, stacked_pred)
    print(f"Training on synthetic: {synth_name}, testing on real data")
    print(classification_report(
        y_real,
        stacked_pred,
        labels=[0, 1, 2, 3],
        target_names=["benign", "attack1", "attack2", "attack3"],
        zero_division=0
    ))
    print(f"F1 Score: {f1:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

    # Sum all training times
    training_time_sec = ocsvm_train_time + lof_train_time + ae_train_time + rf_train_time

    
    def predict_end_to_end(rows):
        rows_scaled = scaler.transform(rows)
        rows_pca_svm_lof = pca_svm_lof.transform(rows_scaled)
        rows_pca_ae = pca_ae.transform(rows_scaled)
        rows_recon = ae.predict(rows_pca_ae)
        rows_stacked = np.vstack([
            ocsvm.decision_function(rows_pca_svm_lof),
            lof.decision_function(rows_pca_svm_lof),
            np.mean((rows_pca_ae - rows_recon) ** 2, axis=1),
        ]).T
        return meta_clf.predict(rows_stacked)

    # Save results
    _row = evaluate(
        y_real,
        stacked_pred,
        model_name=MODEL_NAME,
        family="Synthetic-Stacking",
        y_proba=meta_clf.predict_proba(stacked_features_real) if hasattr(meta_clf, "predict_proba") else None,
        train_time_sec=training_time_sec,
        num_epochs="N/A",
        parameters=str(meta_clf.get_params()),
        timing=time_inference(predict_end_to_end, EXP.X_test_raw[:2000]),
        **EXP.eval_kwargs(),
    )
    print(summary(_row))
    results.append(_row)
    del stacked_features_synth, stacked_features_real
    gc.collect()


In [ ]:
from gps_ids.schema import results_path

append_validated(results, str(results_path("2d")))
print(f"Results appended to {results_path('2d')}")
